In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining"

!pip install -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast
from model import ChessValueNet
from loss_function import WDL_Loss
from dataset_manager import ChunkedChessDataset
from datetime import datetime
import shutil
import os

GPU = "cuda"
EPOCHS = 20

drive_dataset_path = "/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining/train_dataset_chunks"
local_dataset_path = "/content/local_train_chunks"

if not os.path.exists(local_dataset_path):
  shutil.copytree(drive_dataset_path, local_dataset_path)
else:
  pass

device = torch.device(GPU if torch.cuda.is_available() else "cpu")
model = ChessValueNet().to(device)

criterion = WDL_Loss(scaling_factor=4.0)

optimizer = optim.Adam(model.parameters(), lr=0.002)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

scaler = GradScaler(GPU)

dataset = ChunkedChessDataset(folder_path=local_dataset_path)
train_loader = DataLoader(dataset, batch_size=4096, pin_memory=True, num_workers=2)

model.train()
print(f"Starting training loop on device: {device}")

for epoch in range(1, EPOCHS + 1):
  running_loss = 0.0
  batch_count = 0

  for batch_boards, batch_scores in train_loader:
    batch_boards = batch_boards.to(device, non_blocking=True)
    batch_scores = batch_scores.to(device, non_blocking=True).float().view(-1, 1)

    optimizer.zero_grad()

    with autocast(device_type=GPU, dtype=torch.float16):
      predictions = model(batch_boards)
      loss = criterion(predictions, batch_scores)

      scaler.scale(loss).backward()
      scaler.step(optimizer)
      scaler.update()

      running_loss += loss.item()
      batch_count += 1

      if batch_count % 500 == 0:
        print(f"Epoch {epoch} | Batch {batch_count} | Current Loss: {loss.item():.4f} | Time: {datetime.now()}")

  scheduler.step()
  with open("/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining/batch_results.txt", "a") as myfile:
    myfile.write(f"Epoch {epoch} Complete! Average Loss: {running_loss / batch_count:.4f}\n")
  print(f"=== Epoch {epoch} Complete! Average Loss: {running_loss / batch_count:.4f} ===")

save_path = "/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining/chess_model_weights_test.pth"
torch.save(model.state_dict(), save_path)
print("Weights successfully saved to disk!")

Transferring dataset from Google Drive to local NVMe SSD. This may take a minute...
Transfer complete! Commencing high-speed training.
Starting training loop on device: cuda
Epoch 1 | Batch 500 | Current Loss: 0.0119 | Time: 2026-06-18 13:49:28.008951
Epoch 1 | Batch 1000 | Current Loss: 0.0105 | Time: 2026-06-18 13:49:52.062792
Epoch 1 | Batch 1500 | Current Loss: 0.0083 | Time: 2026-06-18 13:50:14.800345
Epoch 1 | Batch 2000 | Current Loss: 0.0087 | Time: 2026-06-18 13:50:38.142987
Epoch 1 | Batch 2500 | Current Loss: 0.0080 | Time: 2026-06-18 13:51:02.136066
Epoch 1 | Batch 3000 | Current Loss: 0.0081 | Time: 2026-06-18 13:51:24.105379
Epoch 1 | Batch 3500 | Current Loss: 0.0073 | Time: 2026-06-18 13:51:47.603850
Epoch 1 | Batch 4000 | Current Loss: 0.0079 | Time: 2026-06-18 13:52:08.967882
Epoch 1 | Batch 4500 | Current Loss: 0.0074 | Time: 2026-06-18 13:52:32.308765
Epoch 1 | Batch 5000 | Current Loss: 0.0073 | Time: 2026-06-18 13:52:55.012477
Epoch 1 | Batch 5500 | Current Loss: 